# `indic/03` — Compute Neural MT Metrics (COMET, BLEURT, BERTScore)

**Purpose:** Score each MT hypothesis in the IndicMT Eval corpus against its
human reference using three neural metrics — COMET, BLEURT, and BERTScore —
under two conditions: native script and romanised script.

**Reads:** `../data/processed/<language>_indicmt.csv`  
(produced by `indic/01_fetch_indicmt_eval.ipynb` and `indic/02_romanisation_pipeline.ipynb`)

**Expected columns (from indic/01 + indic/02):**
- `Source` — English source sentence
- `Reference` — native-script human reference
- `Translation` — native-script MT hypothesis
- `Reference_Transliteration_romanized` — romanised reference (added by indic/02)
- `Translation_Transliteration_romanized` — romanised hypothesis (added by indic/02)

**Outputs produced:**
```
../data/processed/gujarati_indicmt.csv    -- adds columns: comet, bleurt, bertscore_f1,
../data/processed/hindi_indicmt.csv          bertscore_p, bertscore_r (native + _rom)
../data/processed/malayalam_indicmt.csv
../data/processed/marathi_indicmt.csv
../data/processed/tamil_indicmt.csv
../data/mateo/<language>/                -- plain-text files for MATEO upload
../results/<language>_metrics.csv        -- per-language checkpoint CSVs
```

**Two scoring paths:**
- **MATEO (recommended for first use):** Upload per-language files to
  https://mateo.ivdnt.org/Evaluate and download results. Step 2 generates
  files in exactly the format MATEO requires.
- **Local scoring:** Steps 4–6 compute all three metrics directly on CPU or
  GPU using the same model checkpoints.

**Models used:**

| Metric | Model | Version |
|--------|-------|---------|
| COMET | `Unbabel/wmt22-comet-da` | unbabel-comet 2.2.6 |
| BLEURT | `BLEURT-20` | bleurt @ cebe7e6 |
| BERTScore | `microsoft/mdeberta-v3-base` | bert-score 0.3.12 |

---

**References**

- COMET: Rei, R., Stewart, C., Farinha, A. C., & Lavie, A. (2020). COMET: A Neural
  Framework for MT Evaluation. *EMNLP 2020*, pp. 2685–2702.
  https://aclanthology.org/2020.emnlp-main.213

- BLEURT: Sellam, T., Das, D., & Parikh, A. (2020). BLEURT: Learning Robust Metrics
  for Text Generation. *ACL 2020*, pp. 7881–7892.
  https://aclanthology.org/2020.acl-main.704

- BERTScore: Zhang, T., Kishore, V., Wu, F., Weinberger, K. Q., & Artzi, Y. (2020).
  BERTScore: Evaluating Text Generation with BERT. *ICLR 2020*.
  https://arxiv.org/abs/1904.09675

- IndicMT Eval: Sai B., A., Dixit, T., Nagarajan, V., Kunchukuttan, A., Kumar, P.,
  Khapra, M. M., & Dabre, R. (2023). IndicMT Eval: A Dataset to Meta-Evaluate
  Machine Translation Metrics for Indian Languages. *ACL 2023*, pp. 14210–14228.
  https://aclanthology.org/2023.acl-long.795

- MATEO: Vanroy, B., Tezcan, A., & Macken, L. (2023). MATEO: MAchine Translation
  Evaluation Online. *EAMT 2023*, pp. 499–500.
  https://aclanthology.org/2023.eamt-1.52

In [ ]:
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}..."); _install(pkg)

# Neural metric dependencies — install once, then comment out
# Run in this exact order to avoid version conflicts:
#
# !pip install "transformers==4.40.2"
# !pip install "protobuf==4.25.3"
# !pip install "bert-score==0.3.12" --force-reinstall --no-deps
# !pip install git+https://github.com/google-research/bleurt.git@cebe7e6
# !pip install "unbabel-comet==2.2.6"

print("Dependencies ready.")

## Configuration

All paths are relative to the `notebooks/indic/` directory where this notebook lives.

- `DATA_DIR` points to `../data/processed/` — same directory written by `indic/01` and `indic/02`.
- Filenames follow the `<language>_indicmt.csv` convention used by prior notebooks.
- Column names match exactly what `indic/01` and `indic/02` produce.

Set `DEVICE` to `"cuda"` if a GPU is available — COMET and BLEURT benefit
significantly from GPU acceleration at 7,000 sentences.

Set `SCORE_NATIVE` and `SCORE_ROM` to control which conditions are scored.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# protobuf fix: must be set before importing TensorFlow / BLEURT
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

# ── Paths ───────────────────────────────────────────────────────────
# DATA_DIR: where indic/01 and indic/02 write their output CSVs
DATA_DIR    = Path("../../data/processed")
MATEO_DIR   = Path("../../data/mateo")
RESULTS_DIR = Path("../../results")
for d in [MATEO_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Language config ─────────────────────────────────────────────────────
# Keys are the lowercase filename prefixes produced by indic/01
LANG_CONFIGS = {
    "gujarati":  "Gujarati",
    "hindi":     "Hindi",
    "malayalam": "Malayalam",
    "marathi":   "Marathi",
    "tamil":     "Tamil",
}

# ── Column names ─────────────────────────────────────────────────────
# Must match exactly what indic/01 (Source/Reference/Translation)
# and indic/02 (*_Transliteration_romanized) produce.
COL_SRC      = "Source"
COL_HYP      = "Translation"
COL_REF      = "Reference"
COL_HYP_ROM  = "Translation_Transliteration_romanized"
COL_REF_ROM  = "Reference_Transliteration_romanized"

# ── Scoring conditions ───────────────────────────────────────────────────
SCORE_NATIVE = True   # score native-script hyp/ref
SCORE_ROM    = True   # score romanised hyp_rom/ref_rom

# ── Device ───────────────────────────────────────────────────────────
DEVICE = "cpu"  # Options: "cpu" | "cuda"

# ── Model identifiers ────────────────────────────────────────────────────
COMET_MODEL      = "Unbabel/wmt22-comet-da"
BLEURT_CKPT      = "BLEURT-20"
BERTSCORE_MODEL  = "microsoft/mdeberta-v3-base"

print(f"Data directory  : {DATA_DIR.resolve()}")
print(f"MATEO directory : {MATEO_DIR.resolve()}")
print(f"Results dir     : {RESULTS_DIR.resolve()}")
print(f"Device          : {DEVICE}")
print(f"Score native    : {SCORE_NATIVE}")
print(f"Score romanised : {SCORE_ROM}")

## Step 1 — Load Per-Language CSVs

Read the five CSVs written by `indic/01` (with romanised columns added by
`indic/02`). Filenames follow the pattern `<language>_indicmt.csv` with
lowercase language names (e.g. `gujarati_indicmt.csv`).

Confirms that `Source`, `Translation`, `Reference`, `Translation_Transliteration_romanized`,
and `Reference_Transliteration_romanized` are all present before scoring begins.

In [ ]:
data = {}  # language_key -> pd.DataFrame

REQUIRED_NATIVE = [COL_SRC, COL_HYP, COL_REF]
REQUIRED_ROM    = [COL_HYP_ROM, COL_REF_ROM]

for lang_key, lang_name in LANG_CONFIGS.items():
    path = DATA_DIR / f"{lang_key}_indicmt.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found.\n"
            f"Run indic/01_fetch_indicmt_eval.ipynb and indic/02_romanisation_pipeline.ipynb first.")
    df = pd.read_csv(path)
    data[lang_key] = df

    native_ok = all(c in df.columns for c in REQUIRED_NATIVE)
    rom_ok    = all(c in df.columns for c in REQUIRED_ROM)
    print(f"  {lang_name:<12} {len(df):,} rows | "
          f"native cols: {'OK' if native_ok else 'MISSING'} | "
          f"rom cols: {'OK' if rom_ok else 'MISSING (run indic/02)'}")

print(f"\nLoaded {len(data)} language files.")

## Step 2 — Export Files for MATEO

[MATEO](https://mateo.ivdnt.org/) is a web tool that computes BERTScore,
BLEURT, and COMET without local installation. It expects three plain-text
files per language (source / reference / translation), one sentence per
line, no header, UTF-8 encoded.

This step exports those files for both the **native** and **romanised**
conditions into `data/mateo/<language>/native/` and `data/mateo/<language>/romanised/`.

**How to use MATEO:**
1. Go to https://mateo.ivdnt.org/Evaluate
2. Upload `source.txt`, `reference.txt`, and `translation.txt` for one language at a time
3. Select metrics: BERTScore, BLEURT, COMET
4. Click **Evaluate**, download the results CSV
5. Repeat for each language and each condition (native / romanised)
6. Feed the downloaded scores into Step 3 (merge) below

In [ ]:
def export_mateo(df: pd.DataFrame, out_dir: Path,
                 src_col: str, hyp_col: str, ref_col: str) -> None:
    """Write source / translation / reference as plain-text files for MATEO.
    Each file: one sentence per line, no header, UTF-8.
    Both .txt and .tsv versions are written.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    for fname, col in [("source", src_col),
                       ("translation", hyp_col),
                       ("reference", ref_col)]:
        series = df[col].fillna("").astype(str)
        series.to_csv(out_dir / f"{fname}.txt",
                      index=False, header=False, encoding="utf-8")
        series.to_csv(out_dir / f"{fname}.tsv",
                      index=False, header=False, sep="\t", encoding="utf-8")


for lang_key, df in data.items():
    if SCORE_NATIVE and all(c in df.columns for c in [COL_SRC, COL_HYP, COL_REF]):
        out = MATEO_DIR / lang_key / "native"
        export_mateo(df, out, COL_SRC, COL_HYP, COL_REF)
        print(f"  {lang_key} native    -> {out}/")

    if SCORE_ROM and all(c in df.columns for c in [COL_SRC, COL_HYP_ROM, COL_REF_ROM]):
        out = MATEO_DIR / lang_key / "romanised"
        export_mateo(df, out, COL_SRC, COL_HYP_ROM, COL_REF_ROM)
        print(f"  {lang_key} romanised -> {out}/")

print(f"\nMATEO files ready in: {MATEO_DIR.resolve()}")
print("Upload each subfolder separately to https://mateo.ivdnt.org/Evaluate")

## Step 3 — Merge Pre-computed MATEO Scores (Optional)

If you ran scoring via MATEO, download the results CSVs and place them at:
```
../data/mateo/<language>/native/mateo_results.csv
../data/mateo/<language>/romanised/mateo_results.csv
```

Run this cell to merge the scores into the per-language DataFrames.
Skip this step if you are computing metrics locally (Steps 4–6).

Expected MATEO output columns: `COMET`, `BLEURT`, `BERTScore`
(adjust `COL_MAP` below if your MATEO version uses different names).

In [ ]:
# Column name mapping from MATEO output to our canonical names
COL_MAP = {
    "COMET":      "comet",
    "BLEURT":     "bleurt",
    "BERTScore":  "bertscore_f1",
}

for lang_key in LANG_CONFIGS:
    df = data[lang_key].copy()

    for condition in ["native", "romanised"]:
        results_path = MATEO_DIR / lang_key / condition / "mateo_results.csv"
        if not results_path.exists():
            print(f"  [{lang_key} {condition}] mateo_results.csv not found -- skipped")
            continue

        scores = pd.read_csv(results_path)
        suffix = "" if condition == "native" else "_rom"

        for mateo_col, canon_col in COL_MAP.items():
            if mateo_col in scores.columns:
                df[f"{canon_col}{suffix}"] = scores[mateo_col].values

        print(f"  [{lang_key} {condition}] merged {list(COL_MAP.values())} (suffix='{suffix}')")

    data[lang_key] = df

print("\nMATEO merge complete (skipped languages had no results file).")

## Step 4 — Local Scoring: COMET

Compute COMET scores locally using `Unbabel/wmt22-comet-da` (standard DA
model, unbabel-comet 2.2.6). COMET requires source + hypothesis + reference
for each sentence.

Scores are computed for all five languages in sequence. An intermediate
checkpoint CSV is saved after each language so progress is not lost if
a later step fails.

**CPU runtime:** ~20–40 min for 7,000 × 2 conditions.  
**GPU (CUDA):** ~3–6 min. Set `DEVICE = "cuda"` in Configuration.

In [ ]:
from comet import download_model, load_from_checkpoint

gpus = 0 if DEVICE == "cpu" else 1

print(f"Downloading / loading COMET model: {COMET_MODEL}")
comet_path  = download_model(COMET_MODEL)
comet_model = load_from_checkpoint(comet_path)
print("COMET model ready.\n")


def score_comet(df: pd.DataFrame,
                src_col: str, hyp_col: str, ref_col: str) -> list:
    """Return a list of COMET segment scores (0–100 scale)."""
    records = [
        {"src": str(s), "mt": str(h), "ref": str(r)}
        for s, h, r in zip(df[src_col], df[hyp_col], df[ref_col])
    ]
    output = comet_model.predict(records, batch_size=64, gpus=gpus)
    # wmt22-comet-da returns scores in [0, 1]; multiply by 100 to match
    # the 0–100 scale used throughout this pipeline
    return [round(s * 100, 4) for s in output.scores]


for lang_key, df in data.items():
    lang_name = LANG_CONFIGS[lang_key]
    print(f"{'='*55}")
    print(f"{lang_name} — COMET")
    print(f"{'='*55}")

    if SCORE_NATIVE and all(c in df.columns for c in [COL_SRC, COL_HYP, COL_REF]):
        if "comet" not in df.columns:
            df["comet"] = score_comet(df, COL_SRC, COL_HYP, COL_REF)
            print(f"  native    mean COMET = {df['comet'].mean():.2f}")
        else:
            print(f"  native    'comet' already present -- skipped")

    if SCORE_ROM and all(c in df.columns for c in [COL_SRC, COL_HYP_ROM, COL_REF_ROM]):
        if "comet_rom" not in df.columns:
            df["comet_rom"] = score_comet(df, COL_SRC, COL_HYP_ROM, COL_REF_ROM)
            print(f"  romanised mean COMET = {df['comet_rom'].mean():.2f}")
        else:
            print(f"  romanised 'comet_rom' already present -- skipped")

    data[lang_key] = df
    ckpt = RESULTS_DIR / f"{lang_key}_metrics.csv"
    df.to_csv(ckpt, index=False)
    print(f"  [checkpoint] {ckpt}")
    print()

## Step 5 — Local Scoring: BLEURT

Compute BLEURT scores using the `BLEURT-20` checkpoint (~1.2 GB).
The model is downloaded automatically on first run. Manual download:
https://storage.googleapis.com/bleurt-oss-21/BLEURT-20.zip

**Reference:** Sellam, T., Das, D., & Parikh, A. (2020). BLEURT: Learning
Robust Metrics for Text Generation. *ACL 2020*, pp. 7881–7892.
https://aclanthology.org/2020.acl-main.704

In [ ]:
from bleurt import score as bleurt_score

print(f"Loading BLEURT checkpoint: {BLEURT_CKPT}")
bleurt_scorer = bleurt_score.BleurtScorer(BLEURT_CKPT)
print("BLEURT scorer ready.\n")


def score_bleurt(df: pd.DataFrame, hyp_col: str, ref_col: str) -> list:
    """Return a list of BLEURT scores."""
    hyps = df[hyp_col].fillna("").astype(str).tolist()
    refs = df[ref_col].fillna("").astype(str).tolist()
    scores = bleurt_scorer.score(references=refs, candidates=hyps, batch_size=64)
    return [round(s, 4) for s in scores]


for lang_key, df in data.items():
    lang_name = LANG_CONFIGS[lang_key]
    print(f"{'='*55}")
    print(f"{lang_name} — BLEURT")
    print(f"{'='*55}")

    if SCORE_NATIVE and all(c in df.columns for c in [COL_HYP, COL_REF]):
        if "bleurt" not in df.columns:
            df["bleurt"] = score_bleurt(df, COL_HYP, COL_REF)
            print(f"  native    mean BLEURT = {df['bleurt'].mean():.4f}")
        else:
            print(f"  native    'bleurt' already present -- skipped")

    if SCORE_ROM and all(c in df.columns for c in [COL_HYP_ROM, COL_REF_ROM]):
        if "bleurt_rom" not in df.columns:
            df["bleurt_rom"] = score_bleurt(df, COL_HYP_ROM, COL_REF_ROM)
            print(f"  romanised mean BLEURT = {df['bleurt_rom'].mean():.4f}")
        else:
            print(f"  romanised 'bleurt_rom' already present -- skipped")

    data[lang_key] = df
    ckpt = RESULTS_DIR / f"{lang_key}_metrics.csv"
    df.to_csv(ckpt, index=False)
    print(f"  [checkpoint] {ckpt}")
    print()

## Step 6 — Local Scoring: BERTScore

Compute BERTScore (P / R / F1) using `microsoft/mdeberta-v3-base` (~900 MB).
`model_type` is specified explicitly to bypass an AutoModel resolution issue
in `transformers >= 4.41`.

**Reference:** Zhang, T., Kishore, V., Wu, F., Weinberger, K. Q., & Artzi, Y.
(2020). BERTScore: Evaluating Text Generation with BERT. *ICLR 2020*.
https://arxiv.org/abs/1904.09675

In [ ]:
from bert_score import score as bertscore_fn

print(f"BERTScore model : {BERTSCORE_MODEL}")
print(f"Device          : {DEVICE}\n")


def score_bertscore(df: pd.DataFrame,
                    hyp_col: str, ref_col: str) -> tuple:
    """Return (P, R, F1) lists of BERTScore values."""
    hyps = df[hyp_col].fillna("").astype(str).tolist()
    refs = df[ref_col].fillna("").astype(str).tolist()
    P, R, F1 = bertscore_fn(
        cands=hyps,
        refs=refs,
        model_type=BERTSCORE_MODEL,
        verbose=True,
        batch_size=64,
        device=DEVICE,
    )
    return (
        [round(v, 4) for v in P.tolist()],
        [round(v, 4) for v in R.tolist()],
        [round(v, 4) for v in F1.tolist()],
    )


for lang_key, df in data.items():
    lang_name = LANG_CONFIGS[lang_key]
    print(f"{'='*55}")
    print(f"{lang_name} — BERTScore")
    print(f"{'='*55}")

    if SCORE_NATIVE and all(c in df.columns for c in [COL_HYP, COL_REF]):
        if "bertscore_f1" not in df.columns:
            P, R, F1 = score_bertscore(df, COL_HYP, COL_REF)
            df["bertscore_p"]  = P
            df["bertscore_r"]  = R
            df["bertscore_f1"] = F1
            print(f"  native    mean F1 = {df['bertscore_f1'].mean():.4f}")
        else:
            print(f"  native    'bertscore_f1' already present -- skipped")

    if SCORE_ROM and all(c in df.columns for c in [COL_HYP_ROM, COL_REF_ROM]):
        if "bertscore_f1_rom" not in df.columns:
            P, R, F1 = score_bertscore(df, COL_HYP_ROM, COL_REF_ROM)
            df["bertscore_p_rom"]  = P
            df["bertscore_r_rom"]  = R
            df["bertscore_f1_rom"] = F1
            print(f"  romanised mean F1 = {df['bertscore_f1_rom'].mean():.4f}")
        else:
            print(f"  romanised 'bertscore_f1_rom' already present -- skipped")

    data[lang_key] = df
    ckpt = RESULTS_DIR / f"{lang_key}_metrics.csv"
    df.to_csv(ckpt, index=False)
    print(f"  [checkpoint] {ckpt}")
    print()

## Step 7 — Save Updated CSVs

Write all metric columns back into the per-language CSVs in `../data/processed/`.
All downstream notebooks (`indic/04_tokenization_parity.ipynb` onwards)
will find the metric columns alongside the text columns they already contain.

In [ ]:
METRIC_COLS = [
    "comet",          "comet_rom",
    "bleurt",         "bleurt_rom",
    "bertscore_f1",   "bertscore_f1_rom",
    "bertscore_p",    "bertscore_p_rom",
    "bertscore_r",    "bertscore_r_rom",
]

for lang_key, df in data.items():
    out_path = DATA_DIR / f"{lang_key}_indicmt.csv"
    df.to_csv(out_path, index=False)
    present = [c for c in METRIC_COLS if c in df.columns]
    print(f"  Saved  {out_path}  ({len(df):,} rows)  metric cols: {present}")

print(f"\n  {len(data)} files updated in {DATA_DIR.resolve()}")

## Step 8 — Scoring Summary

Print mean COMET, BLEURT, and BERTScore-F1 for each language under both
conditions.

In [ ]:
print(f"{'Lang':<12}  {'Cond':10s}  {'COMET':>8s}  {'BLEURT':>8s}  {'BS-F1':>8s}")
print("-" * 58)

for lang_key, df in data.items():
    lang_name = LANG_CONFIGS[lang_key]
    for cond, suffix in [("native", ""), ("romanised", "_rom")]:
        comet_col = f"comet{suffix}"
        blrt_col  = f"bleurt{suffix}"
        bs_col    = f"bertscore_f1{suffix}"

        comet_m = f"{df[comet_col].mean():.2f}"  if comet_col in df.columns else "--"
        blrt_m  = f"{df[blrt_col].mean():.4f}"   if blrt_col  in df.columns else "--"
        bs_m    = f"{df[bs_col].mean():.4f}"      if bs_col    in df.columns else "--"

        print(f"  {lang_name:<10}  {cond:10s}  {comet_m:>8s}  {blrt_m:>8s}  {bs_m:>8s}")

print("\n  ../data/processed/ is ready for indic/04_tokenization_parity.ipynb.")